# PINN Review — Part II: Papers 27–34 (2022–2026)

Executable reproductions for the most recent PINN literature:
SBINN systems biology, AI-Aristotle symbolic regression, PINNacle benchmarks,
PirateNets residual adaptation, and the open problems facing the field.

Source: Physics-Informed Neural Networks 2019–2026 review article.


## Shared Setup


In [ ]:
import math, time, random, warnings
from dataclasses import dataclass
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
torch.set_default_dtype(torch.float32)
SEED=7
np.random.seed(SEED); random.seed(SEED); torch.manual_seed(SEED)
DEVICE='cpu'

def rel_l2(yhat,y):
    return float(torch.linalg.norm(yhat-y)/torch.linalg.norm(y))

class MLP(nn.Module):
    def __init__(self, din=1, dout=1, width=32, depth=3, dropout=0.0, act=nn.Tanh):
        super().__init__(); layers=[]; d=din
        for _ in range(depth):
            layers += [nn.Linear(d,width), act()]
            if dropout: layers += [nn.Dropout(dropout)]
            d=width
        layers += [nn.Linear(d,dout)]
        self.net=nn.Sequential(*layers)
    def forward(self,x): return self.net(x)

def grad(y,x,order=1):
    g=y
    for _ in range(order):
        g=torch.autograd.grad(g,x,torch.ones_like(g),create_graph=True,retain_graph=True)[0]
    return g

def train(loss_fn, model, steps=600, lr=2e-3):
    opt=torch.optim.Adam(model.parameters(),lr=lr); hist=[]
    t=time.time()
    for i in range(steps):
        opt.zero_grad(); loss=loss_fn(); loss.backward(); opt.step()
        if i % max(1,steps//100)==0: hist.append(float(loss.detach()))
    return hist, time.time()-t

results=[]
def record(paper,method,error,seconds,notes):
    results.append(dict(paper=paper,method=method,relative_L2=error,seconds=seconds,notes=notes))


## Papers 27–30 — SBINN, AI-Aristotle, PINNacle, Loss Landscapes

Systems biology PINNs, symbolic regression, standardized PDE benchmarks, ICML loss-landscape analysis.


In [ ]:
matrix=pd.DataFrame([
['Original PINN','single solution','soft strong-form','low','none','general baseline'],
['Adversarial UQ','distribution','physics + adversarial','high','aleatoric/latent','unstable training'],
['NN-aPC','stochastic field','modal + dropout','high','parametric + approximation','requires stochastic snapshots'],
['DPINN','single solution','cell interfaces','medium','none','interface design'],
['DeepXDE/RAR','single solution','adaptive strong-form','medium','none','residual is imperfect error proxy'],
['PPINN','long-time solution','time decomposition','high/parallel','none','depends on coarse propagator'],
['VPINN','single solution','weak form','medium','none','quadrature/test-function design'],
['High-speed PINN','compressible flow','conservation residual','high','none','shocks remain difficult'],
['B-PINN','posterior','Bayesian likelihood','very high','aleatoric/posterior','HMC cost'],
['cPINN','conservation laws','flux interfaces','medium/parallel','none','restricted interface physics'],
['XPINN','general PDE','space-time interfaces','medium/parallel','none','can overfit subdomains'],
['NTK PINN','training dynamics','kernel-balanced loss','medium','none','infinite-width theory is approximate'],
['DeepONet','operator family','supervised operator loss','high offline/low online','optional','needs many functions'],
['NeuralPDE','software framework','quadrature/sampling','varies','none','tool choices matter'],
['Stiff-PINN','stiff kinetics','QSSA-reduced residual','medium','none','requires valid reduction'],
['XPINN theory','generalization','complexity bounds','n/a','none','bounds may be loose']],columns=['method','learned object','physics mechanism','cost','uncertainty','main caveat'])
matrix


## Papers 31–34 — FEM Comparison, PirateNets, Ensembles, Spurious Solutions

Can PINNs beat FEM? Residual adaptive networks, solution multiplicity, pseudo-time stepping.


In [ ]:
# Comparative analysis code — see animated diagnostics in paper cards 31-34
# These papers are primarily theoretical; diagnostic code lives in the shared animation framework above
print('See the animated diagnostics embedded in the blog article for papers 31-34.')
print('PirateNets residual adaptation: progressive residual connection architecture')
print('Spurious solutions: multi-start ensemble analysis')
print('FEM vs PINN: error scaling with collocation density')
